# Customer Churn Prediction with XGBoost on AWS SageMaker

This notebook demonstrates how to train and deploy a machine learning model using **AWS SageMaker**.

## What You'll Learn
- Setting up a SageMaker session
- Loading and preprocessing data
- Training a model using SageMaker's built-in XGBoost
- Deploying a model endpoint
- Making predictions
- Cleaning up resources

---

> [!IMPORTANT]
> ## ⚠️ Where to Run This Notebook
>
> **This notebook is designed for AWS SageMaker Studio**, not for local execution.
>
> **Instructions:**
> 1. Download this notebook to your computer
> 2. Open AWS SageMaker Studio (see AWS_SageMaker_Guide.md for setup)
> 3. Upload this notebook file to SageMaker Studio
> 4. Select kernel: **Python 3 (Data Science)**
> 5. Run cells in order (Shift + Enter)
>
> **Benefits of SageMaker Studio:**
> - ✅ No credentials needed (uses IAM roles)
> - ✅ All libraries pre-installed
> - ✅ Direct S3/AWS service access
>
> See [AWS_SageMaker_Guide.md](AWS_SageMaker_Guide.md) for detailed instructions.

---

## 1. Setup and Imports

First, let's import the necessary libraries and set up our SageMaker session.

In [ ]:
# Install/upgrade sagemaker if needed (uncomment if running locally)
# !pip install -U sagemaker boto3 pandas

In [ ]:
import sagemaker
import boto3
import pandas as pd
import numpy as np
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator

print(f"SageMaker SDK version: {sagemaker.__version__}")

In [ ]:
# Initialize SageMaker session
sagemaker_session = sagemaker.Session()

# Get the execution role
role = get_execution_role()

# Get default bucket
bucket = sagemaker_session.default_bucket()
prefix = 'sagemaker/xgboost-churn'

# Get region
region = boto3.Session().region_name

print(f"Region: {region}")
print(f"Role: {role}")
print(f"Bucket: {bucket}")

## 2. Load and Explore the Data

We'll use a telecom customer churn dataset. This dataset contains customer information and whether they churned (left the service).

In [1]:
# Download the churn dataset
!wget -q https://raw.githubusercontent.com/aws/amazon-sagemaker-examples/main/introduction_to_applying_machine_learning/xgboost_customer_churn/churn.txt -O churn.txt

# Load the data
df = pd.read_csv("churn.txt")
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Explore the data
print("Dataset Info:")
print(df.info())
print("\nChurn Distribution:")
print(df['Churn?'].value_counts())

In [ ]:
# Basic statistics
df.describe()

## 3. Data Preprocessing

Prepare the data for XGBoost training:
- Convert categorical variables
- Handle the target variable
- Split into train/validation/test sets

In [ ]:
# Convert target to binary
df['Churn?'] = df['Churn?'].map({'True.': 1, 'False.': 0})

# Convert categorical variables
df["Int'l Plan"] = df["Int'l Plan"].map({'yes': 1, 'no': 0})
df['VMail Plan'] = df['VMail Plan'].map({'yes': 1, 'no': 0})

# One-hot encode State column
df = pd.get_dummies(df, columns=['State'], prefix='State')

# Drop columns we don't need
df = df.drop(['Phone', 'Area Code'], axis=1)

print(f"Processed shape: {df.shape}")
df.head()

In [ ]:
# Split the data
from sklearn.model_selection import train_test_split

# First split: 80% train+val, 20% test
train_val, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Churn?'])

# Second split: 80% train, 20% validation (from train_val)
train, val = train_test_split(train_val, test_size=0.2, random_state=42, stratify=train_val['Churn?'])

print(f"Train: {len(train)}, Validation: {len(val)}, Test: {len(test)}")

In [ ]:
# XGBoost expects the target column to be first
def prepare_for_xgboost(df):
    """Move target column to first position and return as numpy array."""
    cols = df.columns.tolist()
    cols.remove('Churn?')
    return df[['Churn?'] + cols].values

train_data = prepare_for_xgboost(train)
val_data = prepare_for_xgboost(val)
test_data = prepare_for_xgboost(test)

print(f"Train data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"Test data shape: {test_data.shape}")

## 4. Upload Data to S3

SageMaker training jobs read data from S3. Let's upload our prepared datasets.

In [ ]:
# Save data locally as CSV
np.savetxt('train.csv', train_data, delimiter=',', fmt='%g')
np.savetxt('validation.csv', val_data, delimiter=',', fmt='%g')
np.savetxt('test.csv', test_data, delimiter=',', fmt='%g')

print("Data saved locally.")

In [ ]:
# Upload to S3
train_s3_path = sagemaker_session.upload_data(
    path='train.csv', 
    bucket=bucket, 
    key_prefix=f'{prefix}/train'
)

val_s3_path = sagemaker_session.upload_data(
    path='validation.csv', 
    bucket=bucket, 
    key_prefix=f'{prefix}/validation'
)

print(f"Train data: {train_s3_path}")
print(f"Validation data: {val_s3_path}")

## 5. Configure and Train the Model

Now we'll set up the XGBoost estimator and launch a training job.

In [ ]:
# Get the XGBoost container image
container = sagemaker.image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='1.5-1'
)

print(f"XGBoost container: {container}")

In [ ]:
# Configure the estimator
xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',  # Use ml.m5.large for lower cost
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=sagemaker_session,
    base_job_name='xgboost-churn'
)

# Set hyperparameters
xgb.set_hyperparameters(
    objective='binary:logistic',
    num_round=100,
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    eval_metric='auc'
)

print("Estimator configured.")

In [ ]:
# Configure training inputs
train_input = TrainingInput(
    s3_data=train_s3_path,
    content_type='text/csv'
)

val_input = TrainingInput(
    s3_data=val_s3_path,
    content_type='text/csv'
)

In [ ]:
# Start training!
# This will take ~5-10 minutes
print("Starting training job...")
xgb.fit({'train': train_input, 'validation': val_input})
print("Training complete!")

## 6. Evaluate the Model

Let's check the training metrics and model performance.

In [ ]:
# Get training job details
training_job_name = xgb.latest_training_job.name
print(f"Training job: {training_job_name}")

# Get model artifact location
model_artifact = xgb.model_data
print(f"Model artifact: {model_artifact}")

In [ ]:
# Get training metrics from CloudWatch
from sagemaker.analytics import TrainingJobAnalytics

metrics = TrainingJobAnalytics(
    training_job_name=training_job_name,
    metric_names=['train:auc', 'validation:auc']
)

metrics.dataframe()

## 7. Deploy the Model

Deploy the trained model to a real-time endpoint for predictions.

> ⚠️ **Warning**: Endpoints incur charges while running. Remember to delete the endpoint when done!

In [ ]:
# Deploy the model
print("Deploying model... (this may take 5-10 minutes)")
from sagemaker.serializers import CSVSerializer

predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    serializer=CSVSerializer()
)

print(f"Endpoint deployed: {predictor.endpoint_name}")

## 8. Make Predictions

Test the endpoint with sample data.

In [ ]:
# Test with a few samples (excluding the target column)
test_samples = test_data[:10, 1:]  # First 10 samples, features only

# Make predictions
predictions = predictor.predict(test_samples)
print("Predictions (probabilities):")
print(predictions.decode('utf-8'))

In [ ]:
# Parse predictions and compare with actual
pred_probs = np.array([float(p) for p in predictions.decode('utf-8').strip().split('\n')])
pred_labels = (pred_probs > 0.5).astype(int)
actual_labels = test_data[:10, 0].astype(int)

results = pd.DataFrame({
    'Actual': actual_labels,
    'Predicted': pred_labels,
    'Probability': pred_probs
})

print("\nPrediction Results:")
print(results)

In [ ]:
# Evaluate on full test set
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Make predictions on all test data
all_predictions = predictor.predict(test_data[:, 1:])
all_probs = np.array([float(p) for p in all_predictions.decode('utf-8').strip().split('\n')])
all_preds = (all_probs > 0.5).astype(int)

# Calculate metrics
accuracy = accuracy_score(test_data[:, 0], all_preds)
auc = roc_auc_score(test_data[:, 0], all_probs)

print(f"\nTest Set Accuracy: {accuracy:.4f}")
print(f"Test Set AUC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(test_data[:, 0], all_preds, target_names=['No Churn', 'Churn']))

## 9. Cleanup Resources ⚠️

**IMPORTANT**: Delete the endpoint to stop incurring charges!

In [ ]:
# DELETE THE ENDPOINT
predictor.delete_endpoint()
print("✅ Endpoint deleted successfully!")

In [ ]:
# Clean up local files
import os

for f in ['train.csv', 'validation.csv', 'test.csv', 'churn.txt']:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted: {f}")

print("\n✅ Local cleanup complete!")

In [ ]:
# Optional: Clean up S3 data
# Uncomment and run if you want to delete the S3 data as well

# s3 = boto3.resource('s3')
# bucket_obj = s3.Bucket(bucket)
# bucket_obj.objects.filter(Prefix=prefix).delete()
# print(f"✅ Deleted S3 data at s3://{bucket}/{prefix}/")

---

## Summary

In this notebook, we:

1. ✅ Set up a SageMaker session
2. ✅ Loaded and preprocessed customer churn data
3. ✅ Uploaded data to S3
4. ✅ Trained an XGBoost model using SageMaker
5. ✅ Deployed the model to a real-time endpoint
6. ✅ Made predictions and evaluated performance
7. ✅ Cleaned up resources

### Next Steps

- Try **Hyperparameter Tuning** to improve model performance
- Explore **SageMaker Pipelines** for MLOps workflows
- Use **SageMaker Model Registry** for model versioning
- Try **Batch Transform** for large-scale batch predictions